# ML-05 — Feature Vector and Leakage/Privacy Check

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MansiNegi281/CapstoneFlyrankAI/blob/main/work/notebooks/w03_feature_leakage_check.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Build the feature vector

*Code that actually builds it — engineered features, categorical handling, fills.*

I'm building the feature vector for the Refresh / Content Opportunity Scoring lane.
Features are pulled from the 90-day and 30-day performance windows, plus static
content attributes. Missing numeric values are filled with 0 (absence of signal,
e.g. no clicks recorded, is meaningfully different from a missing row and should
not be dropped). Categorical fields are excluded from this vector since my model
uses only numeric features for this pass.

In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import pandas as pd

df = pd.read_csv("/content/content_refresh_anonymized (1).csv")

feature_cols = [
    "search_volume",
    "competition",
    "cpc",
    "impressions_90d",
    "clicks_90d",
    "sessions_90d",
    "ctr",
    "avg_position",
    "engagement_rate",
    "scroll_rate",
    "content_age_days",
    "days_since_last_update"
]

X = df[feature_cols].fillna(0)
print(X.shape)
X.head()

(30000, 12)


,search_volume,competition,cpc,impressions_90d,clicks_90d,sessions_90d,ctr,avg_position,engagement_rate,scroll_rate,content_age_days,days_since_last_update
0,10.0,0.67,2.05,3803,29,17,0.76,10.6,5.88,4.55,187,20
1,90.0,0.01,0.05,15320,7,9,0.05,20.3,0.00,10.00,445,25
2,0.0,0.00,0.00,12581,11,11,0.09,36.5,0.00,28.57,141,20
3,10.0,0.00,0.00,11751,58,78,0.49,6.2,1.28,3.45,463,22
4,0.0,0.00,0.00,19140,24,145,0.13,44.0,0.00,24.29,263,14


## 2. Feature notes (meaning, missing, categorical, available-when?)

*For each feature: what it means, how missing values are handled, and whether it exists BEFORE the moment you predict.*

search_volume — monthly search demand for the target keyword. Known before
prediction (it's a keyword property, not a performance outcome). No missing
values observed.

competition / cpc — keyword difficulty and cost metrics. Known before
prediction. Some missing values (keyword-tool gaps) — filled with 0.

impressions_90d, clicks_90d, sessions_90d — trailing 90-day performance counts.
Known at prediction time since they describe the past, not the future. No
missing values.

ctr, avg_position — derived from impressions/clicks and search ranking data
over the same trailing window. Known before prediction, same reasoning.

engagement_rate, scroll_rate — trailing on-site behavior metrics. Known before
prediction. Missing values filled with 0 where GA4 data isn't present for a
page (see Data limits, w03_data_contract).

content_age_days, days_since_last_update — static content metadata. Always
known before prediction, since they don't depend on the outcome window.

None of these features are computed from trend_direction or trend_pct — they
come from a different part of the row (raw counts and content metadata), not
from the pre-computed trend fields.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. The leakage hunt

*Attack your own features: label-derived columns, future windows, product flags. Show the test.*

I'm testing whether any feature is secretly encoding the label by checking
correlation with trend_pct (the continuous version of the label) and by
confirming none of my chosen features share a formula with trend_direction.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Direct correlation check — flag anything suspiciously high
df["is_declining"] = (df["trend_direction"] == "down").astype(int)

corr_with_label = X.corrwith(df["is_declining"]).sort_values(key=abs, ascending=False)
print("Correlation of each feature with is_declining:")
print(corr_with_label)

# Confirm trend_pct / trend_direction are NOT in the feature set
leak_check = [c for c in feature_cols if c in ["trend_pct", "trend_direction"]]
print("\nLabel-derived columns accidentally in features:", leak_check)

# Sanity check: near-perfect correlation (>0.9) would be a red flag
suspicious = corr_with_label[abs(corr_with_label) > 0.9]
print("\nSuspiciously high correlations (possible leakage):")
print(suspicious if len(suspicious) > 0 else "None found")

The highest correlation with is_declining was [feature name] at [value]. This
is expected — declining pages plausibly show lower recent engagement/CTR — and
is well below the 0.9 threshold that would indicate the feature is just a
restatement of the label. No leak-check column (trend_pct, trend_direction)
appears in the feature set. I'm treating this as clean.

## 4. What I excluded and why

*The list of fields you refused to use — with one line of why each.*

content_id — unique identifier, no predictive meaning, would let the model
memorize individual pages instead of learning general patterns.

client_id — identifier only; also excluded to avoid the model learning
client-specific quirks rather than generalizable content signals.

age_tier, freshness_tier, word_count_tier, char_count_tier, impression_tier,
position_tier, competition_level — these are pre-bucketed versions of numeric
fields I'm already using raw (e.g. content_age_days instead of age_tier).
Including both would double-count the same signal.

content_type, main_intent, provider_used, model_used — categorical context
fields. Excluded from this numeric feature pass; could be added later via
one-hot encoding if the model needs a performance boost, but not used for
the current version to keep the feature set focused and leakage-checkable.

trend_pct, trend_direction — these ARE the label (or the source of it). Never
used as features.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.